# 第9章　奇异值分解（SVD） —— 王牌压缩工具

> **本章目标**：掌握 SVD 公式 **A** = **U**Σ**V**ᵀ，用奇异值做图像压缩，理解 LoRA 大模型微调原理。
> **前置知识**：第 6 章（矩阵乘法）、第 8 章（特征值直觉）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("✅ 环境就绪")

---
## 9.1　公式拆解——把任意矩阵拆成三件套

📐 **定义　SVD**：`**A** = **U** @ Σ @ **V**ᵀ`

In [ ]:
np.random.seed(42); A=np.random.randn(5,4)
U,S,Vt=np.linalg.svd(A,full_matrices=False)
print(f"U:{U.shape} S:{S} Vt:{Vt.shape}")
Sigma=np.diag(S); A_recon=U@Sigma@Vt
print(f"重构误差: {np.max(np.abs(A-A_recon)):.2e}  完美✅")

---
## 9.2　奇异值 Σ 的意义——能量的排序

In [ ]:
np.random.seed(42); _,S,_=np.linalg.svd(np.random.randn(20,30),full_matrices=False)
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,4))
ax1.plot(S,'o-',markersize=4); ax1.set_title('Singular Value Decay'); ax1.grid(alpha=0.3)
energy=np.cumsum(S**2)/np.sum(S**2)
ax2.plot(energy,'o-',markersize=4); ax2.axhline(0.9,color='red',ls='--',label='90%')
ax2.set_title('Cumulative Energy'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
k90=np.searchsorted(energy,0.9)+1
print(f"前 {k90} 个奇异值捕获 90% 能量（共 {len(S)} 个）")

---
## 9.3　图像压缩实战

📐 **低秩近似**：取前 k 个奇异值重构，Frobenius 范数下的最优 k 秩近似。

In [ ]:
np.random.seed(0); h,w=200,300; img=np.zeros((h,w))
for _ in range(5):
    cx,cy=np.random.randint(0,w),np.random.randint(0,h)
    y,x=np.ogrid[:h,:w]; img+=np.exp(-((x-cx)**2+(y-cy)**2)/500)
U,S,Vt=np.linalg.svd(img,full_matrices=False)
fig,axes=plt.subplots(2,3,figsize=(12,8))
for idx,k in enumerate([1,5,10,20,50,100]):
    recon=U[:,:k]@np.diag(S[:k])@Vt[:k,:]
    comp=(1-(U[:,:k].size+S[:k].size+Vt[:k,:].size)/img.size)*100
    axes[idx//3,idx%3].imshow(recon,cmap='gray')
    axes[idx//3,idx%3].set_title(f'k={k} ({comp:.0f}% comp)')
    axes[idx//3,idx%3].axis('off')
plt.tight_layout(); plt.show()

---
## 9.4　AI 连接：LoRA 的数学原理

📐 **LoRA（Low-Rank Adaptation）**：ΔW = **B** @ **A**，秩 r ≪ d。只需训练 2dr 个参数。

In [ ]:
d,r=1000,8
W=np.random.randn(d,d)*.01; B=np.random.randn(d,r)*.01; A2=np.random.randn(r,d)*.01
print(f"原始参数: {d*d:,}  LoRA参数: {d*r*2:,}  压缩: {d*d/(d*r*2):.0f}x")
x=np.random.randn(d)
y_orig=W@x; y_lora=(W+B@A2)@x
print(f"LoRA输出与原始差异: {np.max(np.abs(y_lora-y_orig)):.6f}")

---
## ✏️ 习题

> 在下方新建代码单元格作答。

1. （概念）SVD 将矩阵分解为哪三个部分？
2. （概念）为什么前 k 个奇异值能用来压缩数据？
3. （代码）生成 50×40 随机矩阵，对 k=1..min(m,n) 计算重构误差，画衰减曲线。
4. （代码）用随机图案，分别用 k=5,10,20,50 压缩，观察视觉质量变化。

---

> 🔗 **章末钩子**：线性代数核心武器已全部掌握。神经网络凭什么能从数据中"学习"？答案藏在变化率中。
> 预览：下一章——**导数**。

> 💡 **提示**：完成后运行 `Kernel → Restart & Run All` 验证。